<a href="https://colab.research.google.com/github/Hem1144/AI-ML/blob/main/classificationUsingLogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
#initialize SparkSession and installed Required Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("LinearRegression_spark") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()


spark

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
# iris = spark.read.csv("/content/drive/MyDrive/UEL/bezdekIris.data",inferSchema=True, header =True).toDF("sep_len", "sep_wid", "pet_len", "pet_wid", "label")
# iris.select('label').distinct().show(10)
# iris.count()

+---------------+
|          label|
+---------------+
| Iris-virginica|
|    Iris-setosa|
|Iris-versicolor|
+---------------+



149

In [22]:
# from pyspark.ml.feature import VectorAssembler, StringIndexer

# # Convert text labels to numeric indices
# label_indexer = StringIndexer(inputCol="label", outputCol="label_index").fit(iris)
# iris = label_indexer.transform(iris)

# # Combine features into one vector
# vector_assembler = VectorAssembler(
#     inputCols=["sep_len", "sep_wid", "pet_len", "pet_wid"],
#     outputCol="features"
# )
# iris = vector_assembler.transform(iris)

# # Split data (70% train, 30% test)
# train_data, test_data = iris.randomSplit([0.7, 0.3], seed=42)

In [23]:
# from pyspark.ml.classification import DecisionTreeClassifier, LogisticRegression

# # 1. Decision Tree
# dt = DecisionTreeClassifier(labelCol="label_index", featuresCol="features")
# dt_model = dt.fit(train_data)

# # 2. Logistic Regression (automatically handles multiclass)
# lr = LogisticRegression(labelCol="label_index", featuresCol="features")
# lr_model = lr.fit(train_data)

# # 3. Multinomial Regression (same as LogisticRegression in PySpark)
# #    - Already implemented above (LogisticRegression handles multinomial)

In [27]:
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# # Function to evaluate and show confusion matrix
# def evaluate_model(model, test_data):
#     # Predictions
#     predictions = model.transform(test_data)

#     # Confusion Matrix
#     pred_vs_label = predictions.select("prediction", "label_index").rdd
#     confusion_matrix = pred_vs_label.map(lambda x: (x, 1)) \
#                                    .countByKey() \
#                                    .items()

#     print("Confusion Matrix:")
#     for (pred, true), count in confusion_matrix:
#         print(f"Predicted: {pred}, Actual: {true} -> Count: {count}")

#     # Calculate accuracy
#     evaluator = MulticlassClassificationEvaluator(
#         labelCol="label_index",
#         predictionCol="prediction",
#         metricName="accuracy"
#     )
#     accuracy = evaluator.evaluate(predictions)
#     print(f"Accuracy: {accuracy:.4f}\n")
#     return predictions

# # Evaluate all models
# print("Decision Tree Evaluation:")
# dt_predictions = evaluate_model(dt_model, test_data)

# print("Logistic Regression Evaluation:")
# lr_predictions = evaluate_model(lr_model, test_data)

In [33]:
dataset = spark.read.csv('/content/drive/MyDrive/UEL/bezdekIris.data',inferSchema=True, header =True)\
.toDF("sep_len", "sep_wid", "pet_len", "pet_wid", "label")
dataset.select('label').distinct().show(10)
dataset.count()

+---------------+
|          label|
+---------------+
| Iris-virginica|
|    Iris-setosa|
|Iris-versicolor|
+---------------+



149

In [34]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler
vector_assembler = VectorAssembler(\
inputCols=["sep_len", "sep_wid", "pet_len", "pet_wid"],\
outputCol="features")
df_temp = vector_assembler.transform(dataset)
df_temp.show(3)

+-------+-------+-------+-------+-----------+-----------------+
|sep_len|sep_wid|pet_len|pet_wid|      label|         features|
+-------+-------+-------+-------+-----------+-----------------+
|    4.9|    3.0|    1.4|    0.2|Iris-setosa|[4.9,3.0,1.4,0.2]|
|    4.7|    3.2|    1.3|    0.2|Iris-setosa|[4.7,3.2,1.3,0.2]|
|    4.6|    3.1|    1.5|    0.2|Iris-setosa|[4.6,3.1,1.5,0.2]|
+-------+-------+-------+-------+-----------+-----------------+
only showing top 3 rows



In [35]:
df = df_temp.drop('sep_len', 'sep_wid', 'pet_len', 'pet_wid')
df.show(3)

+-----------+-----------------+
|      label|         features|
+-----------+-----------------+
|Iris-setosa|[4.9,3.0,1.4,0.2]|
|Iris-setosa|[4.7,3.2,1.3,0.2]|
|Iris-setosa|[4.6,3.1,1.5,0.2]|
+-----------+-----------------+
only showing top 3 rows



In [36]:
from pyspark.ml.feature import StringIndexer
l_indexer = StringIndexer(inputCol="label", outputCol="labelIndex")
df = l_indexer.fit(df).transform(df)

df.select('label','labelIndex').distinct().show(3)

+---------------+----------+
|          label|labelIndex|
+---------------+----------+
|Iris-versicolor|       0.0|
| Iris-virginica|       1.0|
|    Iris-setosa|       2.0|
+---------------+----------+



In [37]:
(trainingData, testData) = df.randomSplit([0.7, 0.3])

In [38]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
dt = DecisionTreeClassifier(labelCol="labelIndex", featuresCol="features",impurity='entropy', maxDepth=4,seed=1234)
model = dt.fit(trainingData)
predictions = model.transform(testData)

In [42]:
# To estimate the accuracy of the prediction, the test error should be computed :
# why we have MulticlassClassificationEvaluator???

evaluator = MulticlassClassificationEvaluator(labelCol="labelIndex", predictionCol="prediction",metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Test accuracy =  " , accuracy)

Test accuracy =   0.9210526315789473


In [43]:
from pyspark.ml.classification import OneVsRest
from pyspark.ml.classification import LogisticRegression
train, test = df.randomSplit([0.7, 0.3], seed = 2018)
lr = LogisticRegression(maxIter=100, \

                        featuresCol="features", \

                        labelCol='labelIndex')
ovr = OneVsRest(classifier=lr, \
                labelCol='labelIndex', \
                featuresCol='features')
#from pyspark.ml import Pipeline
#pipeline_ovr = Pipeline(stages=[vecAssembler, stdScaler, ovr])
#pipelineModel_ovr = pipeline_ovr.fit(trainDF)

ovrModel = ovr.fit(train)
predictionsovr = ovrModel.transform(test)
evaluator = MulticlassClassificationEvaluator(\
labelCol="labelIndex", predictionCol="prediction",\
metricName="accuracy")
accuracy = evaluator.evaluate(predictionsovr)
print("Test accuracy =  " , accuracy)

Test accuracy =   0.9361702127659575
